# IEEE-CIS 支付交易欺诈风险分析

在每天固定人工审核名额的约束下，比较金额规则、群体规则与风险分排序的效果。

数据：IEEE-CIS Fraud Detection，590,540 笔交易，覆盖 182 天。


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import data, evaluation, profiling, rules, scoring

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


## 1. 数据概览


In [2]:
tx = data.add_day(data.load_transactions())
print(f"交易 {len(tx):,} 笔，覆盖 {tx['day'].nunique()} 天，整体欺诈率 {tx['isFraud'].mean():.2%}")
profiling.dataset_overview(tx)


交易 590,540 笔，覆盖 182 天，整体欺诈率 3.50%


,metric,value
0,transactions,590540.0
1,columns_loaded,8.0
2,frauds,20663.0
3,fraud_rate,0.03499
4,first_day,1.0
5,last_day,182.0
6,covered_days,182.0


In [3]:
profiling.missing_rate(tx, top=10)


,column,missing_rate
0,P_emaildomain,0.159949
1,card4,0.002670
2,card6,0.002660
3,TransactionID,0.000000
4,isFraud,0.000000
5,ProductCD,0.000000
6,TransactionAmt,0.000000
7,TransactionDT,0.000000
8,day,0.000000


## 2. 金额规则敏感性

同一规则只改变阈值，观察送审成本与召回率如何变化。


In [4]:
rules.amount_threshold_table(tx)


,rule,threshold,review_count,review_share,daily_review,fraud_caught,recall,false_alarms,precision,lift
0,金额 > 阈值,100,221921,0.375793,1219.346154,8042,0.389198,213879,0.036238,1.035671
1,金额 > 阈值,200,93880,0.158973,515.824176,4143,0.200503,89737,0.044131,1.261240
2,金额 > 阈值,500,22879,0.038743,125.708791,1008,0.048783,21871,0.044058,1.259156
3,金额 > 阈值,1000,7267,0.012306,39.928571,179,0.008663,7088,0.024632,0.703969
4,金额 > 阈值,2000,2036,0.003448,11.186813,42,0.002033,1994,0.020629,0.589559


## 3. 多维风险画像

同时看交易量、欺诈率、提升倍数与欺诈占比；交易量低于 1,000 的分组不纳入结论。


In [5]:
profiles = profiling.profile_all_dimensions(tx)
profiles["ProductCD"]


,transactions,frauds,fraud_rate,lift,fraud_share,daily_transactions
ProductCD,,,,,,
C,68519,8008,0.116873,3.340173,0.387553,376.478022
S,11628,686,0.058996,1.686068,0.033199,63.890110
H,33024,1574,0.047662,1.362169,0.076175,181.450549
R,37699,1426,0.037826,1.081050,0.069012,207.137363
W,439670,8969,0.020399,0.583006,0.434061,2415.769231


In [6]:
profiles["ProductCD_x_card6"]


transactions  frauds  fraud_rate      lift  fraud_share  daily_transactions
ProductCD card6                                                                              
C         credit         27551    4663    0.169250  4.837088     0.225669          151.379121
          debit          40763    3327    0.081618  2.332613     0.161012          223.972527
S         debit           5100     315    0.061765  1.765210     0.015245           28.021978
          credit          6527     371    0.056841  1.624487     0.017955           35.862637
H         credit         17656     878    0.049728  1.421210     0.042491           97.010989
          debit          15367     696    0.045292  1.294423     0.033683           84.434066
W         credit         70753    3006    0.042486  1.214227     0.145477          388.752747
R         credit         26499    1032    0.038945  1.113028     0.049944          145.598901
          debit          11192     394    0.035204  1.006108     0.019068           61.494505
W         debit         367516    5942    0.016168  0.462075     0.287567         2019.318681
          NaN             1371      21    0.015317  0.437762     0.001016            7.532967

In [7]:
profiles["P_emaildomain"]


,transactions,frauds,fraud_rate,lift,fraud_share,daily_transactions
P_emaildomain,,,,,,
outlook.com,5096,482,0.094584,2.703171,0.023327,28.000000
hotmail.com,45250,2396,0.052950,1.513297,0.115956,248.626374
gmail.com,228355,9943,0.043542,1.244408,0.481198,1254.697802
icloud.com,6267,197,0.031434,0.898385,0.009534,34.434066
comcast.net,7888,246,0.031187,0.891300,0.011905,43.340659
NaN,94456,2790,0.029538,0.844171,0.135024,518.989011
bellsouth.net,1909,53,0.027763,0.793462,0.002565,10.489011
live.com,3041,84,0.027622,0.789439,0.004065,16.708791
anonymous.com,36998,859,0.023217,0.663546,0.041572,203.285714


## 4. 交易表与 identity 表连接


In [8]:
identity = data.load_identity()
merged, merge_summary = data.merge_identity(tx, identity)
merge_summary


,check,value
0,transaction_rows,590540
1,identity_rows,144233
2,transaction_id_unique,True
3,identity_id_unique,True
4,merged_rows,590540
5,row_count_preserved,True
6,identity_covered,144233
7,identity_coverage_rate,0.244239


In [9]:
data.identity_join_profile(merged)


,transactions,frauds,fraud_rate
_merge,,,
left_only,446307,9345,0.020939
both,144233,11318,0.078470


In [10]:
data.identity_missing_summary(merged)


,metric,value
0,id_columns,38.0
1,id_01_missing_rate,0.755761
2,median_missing_rate,0.824428
3,max_missing_rate,0.991962


## 5. 时间切分与群体稳定性

训练期第 1–109 天，验证期第 110–145 天，测试期第 146 天之后。


In [11]:
periods = data.split_by_time(tx)
data.describe_periods(periods)


,period,days,transactions,frauds,fraud_rate
0,train,109,380100,12992,0.034180
1,valid,36,104747,4060,0.038760
2,test,37,105693,3611,0.034165


In [12]:
evaluation.segment_stability(periods)


,segment,train_transactions,train_rate,valid_transactions,valid_rate,test_transactions,test_rate
0,产品C + 信用卡,18212,0.163519,5314,0.171810,4025,0.191801
1,产品C + 借记卡,26249,0.069450,7669,0.107967,6845,0.098758
2,W + 借记卡,226981,0.016054,70842,0.016586,69693,0.016114
3,所有信用卡,101533,0.064265,22979,0.081466,24474,0.063455
4,gmail 邮箱,145567,0.042530,41986,0.049374,40802,0.041150
5,hotmail / outlook,33025,0.055564,9152,0.056927,8169,0.063900


## 6. 查表打分与审核容量

打分器只在训练期拟合，按“产品 + 卡类型 + 邮箱 → 产品 + 卡类型 → 整体欺诈率”三级回退。


In [13]:
scorer = scoring.RateLookupScorer().fit(periods["train"])
scored_periods = {name: scorer.transform(periods[name]) for name in ("valid", "test")}


In [14]:
capacity = evaluation.capacity_curve(scored_periods)
capacity[capacity["period"] == "test"]


,period,strategy,capacity,reviews,fraud_caught,precision,recall
10,test,amount,25,925,38,0.041081,0.010523
11,test,score,25,925,294,0.317838,0.081418
12,test,amount,50,1850,87,0.047027,0.024093
13,test,score,50,1850,465,0.251351,0.128773
14,test,amount,100,3700,208,0.056216,0.057602
15,test,score,100,3700,781,0.211081,0.216284
16,test,amount,200,7400,376,0.050811,0.104126
17,test,score,200,7400,1273,0.172027,0.352534
18,test,amount,500,18500,767,0.041459,0.212407
19,test,score,500,18500,1836,0.099243,0.508446


## 7. 分数分档与三档策略


In [15]:
evaluation.score_tier_table(scored_periods)


,period,tier,transactions,daily_transactions,frauds,fraud_rate,fraud_share
0,valid,分数 ≥ 0.20,2383,66.194444,631,0.264792,0.155419
1,valid,0.10 ≤ 分数 < 0.20,6260,173.888889,887,0.141693,0.218473
2,valid,0.05 ≤ 分数 < 0.10,11978,332.722222,738,0.061613,0.181773
3,valid,分数 < 0.05,84126,2336.833333,1804,0.021444,0.444335
4,test,分数 ≥ 0.20,1887,51.000000,472,0.250132,0.130712
5,test,0.10 ≤ 分数 < 0.20,5339,144.297297,819,0.153400,0.226807
6,test,0.05 ≤ 分数 < 0.10,14026,379.081081,640,0.045630,0.177236
7,test,分数 < 0.05,84441,2282.189189,1680,0.019896,0.465245


In [16]:
evaluation.policy_comparison(scored_periods)


,period,policy,rejects_per_day,reject_precision,false_rejects_per_day,reviews_per_day,total_recall
0,valid,A 激进,66.194444,0.264792,48.666667,100.0,0.270690
1,valid,B 保守,0.555556,0.750000,0.138889,100.0,0.194089
2,valid,C 金额门槛,6.972222,0.450199,3.833333,100.0,0.197537
3,test,A 激进,51.000000,0.250132,38.243243,100.0,0.290778
4,test,B 保守,0.540541,0.450000,0.297297,100.0,0.218222
5,test,C 金额门槛,5.189189,0.395833,3.135135,100.0,0.226530


In [17]:
review_window = pd.concat([scored_periods["valid"], scored_periods["test"]], ignore_index=True)
evaluation.weekly_stability(review_window)


,segment,start_day,end_day,days,reviews,fraud_caught,precision,recall
0,1,110,116,7,700,170,0.242857,0.175077
1,2,117,123,7,700,177,0.252857,0.186316
2,3,124,130,7,700,148,0.211429,0.206416
3,4,131,137,7,700,135,0.192857,0.197368
4,5,138,144,7,700,137,0.195714,0.204173
5,6,145,151,7,700,135,0.192857,0.239362
6,7,152,158,7,700,131,0.187143,0.197587
7,8,159,165,7,700,110,0.157143,0.171340
8,9,166,172,7,700,166,0.237143,0.228966
9,10,173,179,7,700,183,0.261429,0.245638


## 8. 结论

1. 整体欺诈率 3.50%，正负样本极不均衡，评估应关注命中率与召回率，而不是准确率。
2. “产品 C + 信用卡”是最显著的风险群体：欺诈率 16.92%，提升 4.84 倍，贡献 22.57% 的欺诈交易。
3. 单条金额规则效果有限：阈值 500 时命中率 4.41%，仅略高于整体欺诈率。
4. 在测试期每天 100 个审核名额下，风险分排序的召回率为 21.63%，金额排序为 5.76%。
5. 七天分段结果显示策略效果基本稳定，可作为规则上线前的基线方案。
